In [ ]:
from src.sklearn_grid_search import *

import pandas as pd
import numpy as np

from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier

In [ ]:
# select device to run this on
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# create synthetic data

n_samples, n_features = 1000, 10                      # number of samples & number of features

x = torch.randn(n_samples, n_features)                # randomly generated features
weights = torch.randn(n_features) * 5                 # randomly generated weights

y = (x @ weights)                                     # linear combinations of random features & weights
y += torch.randn(n_samples) * abs(y).mean() * 0.5     # normal random error

y = nn.Sigmoid()(y).round()                           # sigmoid for binary classification

In [ ]:
# select model
model = RandomForestClassifier(random_state = 18)

# input dictionary corresponding to parameters & ranges to test
param_dict = {
  
              'max_depth':[2, 3, 4, 5, 6, 10, 12, 16, None],
              'max_features': ['sqrt', 'log2', 2, 3, 5, 7, 9, 12, None],
              'n_estimators': [5, 10, 25, 50, 100, 150, 200, 300],

              }

In [ ]:
# run function to approximate best parameter combinations using Sparse Tensor Completion

best_estimated_params= return_best_k_params(model = model,                              # ML model to hyperparameter tune
                                            param_dict = param_dict,                    # dictionary of parameters
                                            X = x,
                                            Y = y,
                                            num_top_combinations = 10,                  # number of best estimated combinations to return
                                            cv_splits = 5,                              # number of Cross Validation folds
                                            tensor_portion = 25,               # fraction/number of combinations computed to estimate all
                                            tensor_completion_model = 'cpd.smooth',     # sparse tensor completion model
                                            metric = 'f1',                              # evaluation metric
                                            rank = 5,                                   # rank decomposition used for tensor completion
                                            device = device,                            # device to run this on
                                            verbose = True)

In [ ]:
# display best estimated parameters and their actual evaluation metric
for p in best_estimated_params: print(f"{p[0]} \nactual eval = {p[1]:.5f}\n")